# 🔬 TruthLens: Image Deepfake Model Training & Calibration
This notebook fine-tunes **EfficientNet-B0** for digital forensics, applies **Social Media JPEG Compression Augmentation**, performs **Platt Temperature Calibration ($T$)**, and exports the checkpoint ready to drop into `TruthLens/backend/models/`.

### **Step 1: Enable Free GPU**
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [1]:
# Install dependencies
!pip install -q torch torchvision datasets scikit-learn pillow

zsh:1: command not found: pip


In [2]:
import os, io, json, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on device:", device)

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Load balanced real vs fake dataset
DATASET_NAME = "JamieWithofs/Deepfake-and-real-images"
print(f"Downloading dataset: {DATASET_NAME}...")
ds = load_dataset(DATASET_NAME)

train_split = ds["train"].shuffle(seed=42)
val_split = ds.get("validation", ds.get("test", train_split.select(range(2000))))
print(f"Train size: {len(train_split)}, Val size: {len(val_split)}")

In [ ]:
# Social media JPEG compression simulation augmentation
class RandomJPEGCompression:
    def __init__(self, quality_min=45, quality_max=90, p=0.4):
        self.quality_min = quality_min
        self.quality_max = quality_max
        self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        q = random.randint(self.quality_min, self.quality_max)
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    RandomJPEGCompression(),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class HFDatasetAdapter(Dataset):
    def __init__(self, split, tf):
        self.split = split
        self.tf = tf
    def __len__(self): return len(self.split)
    def __getitem__(self, idx):
        item = self.split[idx]
        img = item["image"].convert("RGB")
        return self.tf(img), torch.tensor(item["label"], dtype=torch.long)

train_loader = DataLoader(HFDatasetAdapter(train_split, train_tf), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(HFDatasetAdapter(val_split, val_tf), batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Model: EfficientNet-B0 with custom binary head
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
epochs = 5

for epoch in range(1, epochs + 1):
    model.train()
    running_loss, total, correct = 0.0, 0, 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        preds = out.argmax(dim=1)
        total += lbls.size(0)
        correct += (preds == lbls).sum().item()
    print(f"Epoch [{epoch}/{epochs}] Loss: {running_loss/total:.4f} Acc: {correct/total*100:.2f}%")

In [ ]:
# Platt Temperature Scaling on validation logits
model.eval()
val_logits, val_labels = [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        val_logits.append(model(imgs.to(device)).cpu())
        val_labels.append(lbls)
logits = torch.cat(val_logits, dim=0)
labels = torch.cat(val_labels, dim=0)

temp_param = nn.Parameter(torch.ones(1) * 1.0)
opt = optim.LBFGS([temp_param], lr=0.01, max_iter=50)
def eval_step():
    opt.zero_grad()
    loss = nn.CrossEntropyLoss()(logits / temp_param, labels)
    loss.backward()
    return loss
opt.step(eval_step)
temperature_T = float(temp_param.item())
print(f"Optimal Platt Scaling Temperature T = {temperature_T:.4f}")

# Metrics
scaled_probs = torch.softmax(logits / temperature_T, dim=1).numpy()
preds = np.argmax(scaled_probs, axis=1)
y_true = labels.numpy()
acc = float(accuracy_score(y_true, preds))
f1 = float(f1_score(y_true, preds, average='macro'))
auc = float(roc_auc_score(y_true == 0, scaled_probs[:, 0]))
cm = confusion_matrix(y_true, preds).tolist()
print(f"Validation Accuracy: {acc*100:.2f}%, AUC: {auc:.4f}, F1: {f1:.4f}")

In [ ]:
# Export files
os.makedirs("truthlens_output", exist_ok=True)
ckpt_path = "truthlens_output/truthlens_image_effb0.pth"
torch.save({
    "state_dict": model.state_dict(),
    "classes": ["fake", "real"],
    "fake_class_index": 0,
    "temperature_T": round(temperature_T, 4)
}, ckpt_path)

metrics_path = "truthlens_output/model_metrics.json"
with open(metrics_path, "w") as f:
    json.dump({
        "status": "custom_trained",
        "model": "EfficientNet-B0 (TruthLens Custom)",
        "dataset": DATASET_NAME,
        "train_samples": len(train_split),
        "val_samples": len(val_split),
        "classes": ["fake", "real"],
        "fake_class_index": 0,
        "temperature_T": round(temperature_T, 4),
        "val_accuracy": round(acc, 4),
        "f1_score": round(f1, 4),
        "roc_auc": round(auc, 4),
        "confusion_matrix": cm
    }, f, indent=2)

# Download files to your computer
from google.colab import files
files.download(ckpt_path)
files.download(metrics_path)
print("Downloaded! Move these two files into your TruthLens backend/models/ folder.")